# Lab 8: Building and Improving a Document RAG System

In [2]:
import os
from openai import OpenAI

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from IPython.display import display, Markdown

client = OpenAI()

Throughout this unit, you have explored how RAG systems work and built the individual pieces: embeddings and vector similarity, document loaders, chunking strategies, vector indexes, RAG chains, and query transformation techniques. In this lab, you will assemble those pieces into a complete RAG application for an outdoor gear company, then improve its retrieval quality by adding a query rewriting step on top.

You will complete the following tasks:

1. Load and chunk the knowledge base:
    * Load a customer support corpus from a single text file
    * Choose chunking parameters and justify your choice in writing
2. Build the vector index:
    * Initialize an embedding model and a Chroma vector store
    * Inspect retrieval quality on a test query using similarity scores
3. Write the RAG instruction prompt:
    * Write a prompt template that grounds the model in retrieved context and handles unknowns
4. Assemble the baseline RAG chain:
    * Connect the retriever, prompt, LLM, and output parser using the pipe operator
    * Test the chain on three different kinds of customer queries
5. Add query rewriting and compare:
    * Write a query rewriting prompt that expands short customer queries
    * Compare baseline answers to rewritten-query answers on the same three queries
6. Analysis:
    * Evaluate the pipeline's behavior, limitations, and business implications
7. AI Reflection:
    * Reflect on how you used (or did not use) AI tools during this lab

<div style="border:1px solid #ccc; border-radius:8px; padding:12px; background-color:#f8f9fa;">

<p><strong>Important:</strong> All graded cells that require you to enter code will contain regions marked with <code># YOUR CODE HERE</code> and <code># END OF YOUR CODE</code>. Between these lines, you’ll find the line <code>raise NotImplementedError("Your code is missing.")</code>, like so:</p>

```python
# YOUR CODE HERE
raise NotImplementedError("Your code is missing.")
# END OF YOUR CODE
```
<p></p>
<p>These markers indicate exactly where you should enter your answer. Replace the line <code>raise NotImplementedError("Your code is missing.")</code> with your code.

## Business Context

Read through the scenario below. You will be putting yourself in the shoes of a junior ML engineer (MLE) at Marlowe & Finch, where you have been asked to prototype a customer support assistant for the company's website.

#### 1. Company and Context

Marlowe & Finch is a small outdoor gear company based in Boulder, Colorado. The company was founded in 2017 by two former trail crew leads who got tired of gear that looked great in the store and fell apart on the trail. Today Marlowe & Finch designs lightweight, three-season backpacking gear for weekend backpackers and thru-hikers, and sells through its own website and a handful of independent outdoor retailers.

The customer base is mostly enthusiastic but not expert. Many customers are buying their first real piece of backcountry gear and have a lot of practical questions before they pull the trigger on a $400 tent.

#### 2. Business Challenge

Marlowe & Finch's customer support team currently answers most questions by hand, working from an internal knowledge base of product specs, warranty terms, and returns/shipping policies. Response times have been slipping as the volume of presale questions grows, and customers who do not get a quick answer often abandon their carts. The team has been asked to build a prototype assistant that can answer common product and policy questions on the website, using the same knowledge base the human agents reference today.

#### 3. Business Goal

The goal is to build a working RAG (Retrieval-Augmented Generation) prototype that can take a customer's question, retrieve the most relevant passages from the Marlowe & Finch knowledge base, and produce a grounded answer. If the prototype performs well, it would be deployed as a first-line assistant on the website. Customer support agents would handle anything the assistant cannot.

#### 4. Your Role and Task

You have just joined Marlowe & Finch as a junior MLE on the Customer Experience team. The team has handed you the customer-facing knowledge base as a single text file and asked you to build and test a prototype RAG assistant. Your job is to make the chunking, retrieval, and prompting decisions, then evaluate whether the prototype produces answers the customer support team would be comfortable putting in front of customers.

This work requires judgment at every stage. The chunking parameters you pick determine what the retriever can find. The retrieval setup determines what context the model sees. The instruction prompt determines whether the model stays grounded in the retrieved context or wanders into territory the knowledge base does not cover. A small change in any of these stages can meaningfully change the answers customers receive.

#### 5. Technical Focus in This Lab

This lab focuses on building and improving a document RAG pipeline:

* **Document Loading and Chunking** &mdash; Loading a source document and splitting it into focused chunks with `RecursiveCharacterTextSplitter`.
* **Vector Indexing and Retrieval** &mdash; Embedding chunks with OpenAI embeddings and storing them in a Chroma vector store for similarity search.
* **Prompt Engineering for RAG** &mdash; Writing an instruction prompt that grounds the model in retrieved context and handles questions the knowledge base cannot answer.
* **Chain Composition with LangChain** &mdash; Using the pipe operator to assemble a complete RAG chain.
* **Query Transformation** &mdash; Adding a query rewriting step in front of the retriever to improve retrieval on short, vague queries.

## Part 1. Load and Chunk the Knowledge Base

Marlowe & Finch has provided you with a single text file containing the customer-facing knowledge base. It is stored at `data/marlowe_knowledge_base.txt` and includes the company's "About" content, specs for three products, warranty terms, returns and shipping policies, and an FAQ.

Before you can search this knowledge base with embeddings, you need to load it and split it into chunks. You practiced both steps in the LangChain primer and the chunking strategies activity.

**Task**: Use `TextLoader` to load the file at `data/marlowe_knowledge_base.txt`. Save the result to a variable called `documents`. Then print how many documents you loaded and the total character length, so you can sanity-check that the file was read correctly.

*Tip*: `TextLoader(path).load()` returns a list of `Document` objects.

In [3]:
# YOUR CODE HERE
loader = TextLoader("data/marlowe_knowledge_base.txt")
documents=loader.load()

total_chars= sum(len(doc.page_content) for doc in documents)

# sanity check
print(f"Loaded {len(documents)} document(s).")
print(f"Total character length: {total_chars}")
# END OF YOUR CODE

Loaded 1 document(s).
Total character length: 12430


**Task**: Print the first 1,500 characters of the document so you can see what kind of content you are working with. You will use this view to inform your chunking choices in the next step.

*Tip*: You can slice a string with `text[:1500]`. The full document content lives in `documents[0].page_content`.

In [4]:
# YOUR CODE HERE
sample_text = documents[0].page_content[:1500]

print(sample_text)
# END OF YOUR CODE

MARLOWE & FINCH CUSTOMER SUPPORT KNOWLEDGE BASE
Last updated: Spring 2025

Welcome to the Marlowe & Finch customer support knowledge base. This document
is the internal source of truth our support team uses to answer customer
questions. It covers our current product lineup, warranty terms, returns and
shipping policies, and frequently asked questions.

ABOUT MARLOWE & FINCH

Marlowe & Finch is a small outdoor gear company based in Boulder, Colorado.
We design backcountry equipment for weekend backpackers and thru-hikers, with
a focus on lightweight three-season gear that holds up to real use. Our
products are sold through our website and a handful of independent outdoor
retailers. We do not currently sell through Amazon, REI, or other large
marketplaces.

We were founded in 2017 by two former trail crew leads who got tired of gear
that looked great in the store and fell apart on the trail. Every product in
our lineup is field-tested by our own staff before it ships.

PRODUCT CATALOG

-

Now that you have seen the content, take a moment to think about its structure. It mixes brand story, product specs, policy language, and FAQ entries. A chunk that mixes a product spec with an unrelated policy paragraph would be a noisy hit for either kind of query. A chunk that is too small might cut a policy sentence in half so that neither chunk contains a complete answer.

You will now split the document into chunks. The chunking strategies activity covered the trade-offs between small, medium, and large chunks, and the role of overlap.

**Task**: Create a `RecursiveCharacterTextSplitter` with `chunk_size` and `chunk_overlap` values of your choosing. Then call `.split_documents()` on `documents` to produce the chunks. Save the result to a variable called `chunks`.

After running your splitter, print the total number of chunks and the average chunk length so you can sanity-check your choice.

In [5]:
# YOUR CODE HERE

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=50,separators=["\n\n", "\n", " ", ""])

chunks = text_splitter.split_documents(documents)

avg_chunk_length = sum(len(chunk.page_content) for chunk in chunks) / len(chunks)

# sanity-check results
print(f"Total number of chunks: {len(chunks)}")
print(f"Average chunk length: {avg_chunk_length:.2f} characters")
# END OF YOUR CODE

Total number of chunks: 35
Average chunk length: 355.00 characters


**Task**: In the markdown cell below, answer the following:

1. What `chunk_size` and `chunk_overlap` did you choose?
2. Why did you pick those values for this particular knowledge base? Refer to something specific you saw in `documents[0].page_content`.
3. What is one risk of your choice (for example, what kind of customer question might it handle poorly)?

1. My chunk_size was 500 and chunk overlap - 50.
2. Looking at the first 1,500 characters of documents[0].page_content, the knowledge base is organized into short, dense paragraphs containing specific product specifications (like tent weights, dimensions, and materials) and clear policy bullet points. A chunk_size of 500 characters (75–100 words) is large enough to capture an entire product specification or policy paragraph together in a single context window without cutting it off, while keeping the chunks focused enough to avoid diluting semantic embedding scores. The chunk_overlap of 50 characters (about 10%) ensures that key terms at paragraph or sentence boundaries aren't severed or cut off during splitting.
3. If a customer asks a complex several-part or comparative question (like, "compare the weights **AND** pole materials, **AND** warranty coverage across **all** three tent models"), the answer will span across multiple individual chunks and it might be a bit cooked. Because each 500-character chunk only contains details about a single product or policy section, a standard k=3 retriever might miss necessary context for cross-product comparisons.

Now that you have chunks, you will embed them and store them in a vector database so the retriever can find the most relevant chunks for any customer query. You did this same pipeline in the Build a Vector Index activity.

**Task**: Initialize an `OpenAIEmbeddings` object using the `text-embedding-3-small` model. Save it to a variable called `embeddings`. Then build a Chroma vector store from your chunks using `Chroma.from_documents()`. Save the result to a variable called `vectorstore`.

In [6]:
# YOUR CODE HERE
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Chroma.from_documents(documents=chunks,embedding=embeddings)
# END OF YOUR CODE

**Task**: Create a retriever from your vector store using `vectorstore.as_retriever()`. Configure it to use similarity search and to return the top 4 chunks. Save it to a variable called `retriever`.

*Tip*: `as_retriever()` accepts `search_type` and `search_kwargs` arguments. The `k` value goes inside `search_kwargs`.

In [7]:
# YOUR CODE HERE
retriever = vectorstore.as_retriever(search_type="similarity",search_kwargs={"k": 4})
# END OF YOUR CODE

Before you build the full RAG chain, let's inspect what the retriever returns for a real customer query, along with the similarity scores. The cell below uses `similarity_search_with_score()` so you can see how close each retrieved chunk actually is to the query.

In [8]:
test_query = "what's your return policy"

scored_results = vectorstore.similarity_search_with_score(test_query, k=4)

print(f"Query: '{test_query}'")
print(f"\nTop 4 retrieved chunks (lower distance = more similar):\n")
for i, (doc, score) in enumerate(scored_results, 1):
    preview = doc.page_content[:200].replace("\n", " ")
    print(f"--- Chunk {i} (distance: {score:.4f}) ---")
    print(f"{preview}...\n")

Query: 'what's your return policy'

Top 4 retrieved chunks (lower distance = more similar):

--- Chunk 1 (distance: 0.8323) ---
========================================================== RETURNS AND SHIPPING POLICY ==========================================================  -----------------------------------------------------...

--- Chunk 2 (distance: 0.9204) ---
To start a return: 1. Email returns@marloweandfinch.com with your order number. 2. We will email you a prepaid return label within one business day. 3. Pack the item in its original packaging and drop...

--- Chunk 3 (distance: 1.1310) ---
To qualify as "unused," items must be: - In original packaging with all tags attached - Free of dirt, pet hair, scent, and any signs of outdoor use - Free of modifications (no aftermarket patches, sew...

--- Chunk 4 (distance: 1.2200) ---
Q: What's your sustainability policy? A: We are a small company and try to be honest about what we do and do not do on sustainability. Our shell fabrics

**Task**: Look at the output above and think about how well the retriever is working. In the markdown cell below, answer the following:

1. Are the top retrieved chunks actually about the return policy, or are some of them off-topic?
2. What do the distance scores tell you? Is there a clear gap between the top result and the bottom result, or are they similar?
3. If you saw an off-topic chunk in the top 4, what do you think pulled it in? If all four chunks were on-topic, would your answer change if a customer asked something more specific, like "can I return a tent I used once"?


1. **Relevance of Chunks:** 
Chunks 1, 2, and 3 directly cover return details. Chunk 1 shows the policy header, Chunk 2 details return steps, and Chunk 3 lists item condition criteria. Chunk 4 discusses company sustainability measures, so that output sits off-topic.

2. **Distance Score Insights:** 
The distance values range from 0.8323 at top relevance down to 1.2200 at the fourth position. A clear gap exists across this spectrum. The cosine/L2 distance rises steadily with each rank step, marking a sharp quality drop off between the direct return steps in Chunk 2 (0.9204) and the sustainability section in Chunk 4 (1.2200).

3. **Retrieval Triggers and Specificity:** 
The shared term "policy" pulled in Chunk 4 because embedding vectors weigh shared vocabulary alongside semantic meaning. A specific query like "can I return a tent I used once" would shift similarity scores toward condition requirements like Chunk 3, while likely pushing the sustainability chunk out of top results entirely.

## Part 3. Write the RAG Instruction Prompt

You have a retriever. Now you need an instruction prompt that tells the LLM what to do with the retrieved context. You wrote one of these in the Build Your First RAG System activity. In that activity, the prompt was for a generic question-answering assistant. Here you will write one tailored to Marlowe & Finch's customer support tone and to the specific risks of running a customer-facing assistant.

A good RAG prompt for this use case should:

1. Position the model as a customer support assistant for Marlowe & Finch
2. Tell the model to answer using only the retrieved context
3. Tell the model exactly what to say when the context does not contain the answer (for example, that it cannot find that information and the customer should contact customer support at `support@marloweandfinch.com`)
4. Set constraints on tone and length so the answers sound like the support team

**Task**: Create a variable called `rag_instruction` and assign it a prompt template string. The template must include the exact placeholders `{context}` and `{question}` so it can be used with `PromptTemplate.from_template()` later. Write the instruction text yourself, following the four requirements above.

*Tip*: Use triple quotation marks (`"""`) to write a multi-line string.

In [1]:
# YOUR CODE HERE
rag_instruction = """You are a helpful and polite customer support assistant for Marlowe & Finch, a lightweight outdoor gear company based in Boulder, Colorado.

Answer the customer's question using ONLY the provided context below. Follow these guidelines:
1. Tone & Style: Be professional, friendly, clear, and concise. Sound like an helpful customer support agent.
2. Grounding: Strict adherence to the context is required. Do not invent facts, product specs, or policies not explicitly mentioned.
3. Missing Information: If the context does not contain enough information to answer the question, state clearly: "I'm sorry, but I don't have that information available. Please contact our support team at support@marloweandfinch.com for further assistance."

Context:
{context}

Question:
{question}

Answer:"""
# END OF YOUR CODE

The cell below builds a `PromptTemplate` object from the string you wrote and prints the input variables it detected. You should see `['context', 'question']`.

In [9]:
# Do not remove or edit this cell

prompt = PromptTemplate.from_template(rag_instruction)
print("PromptTemplate input variables:", prompt.input_variables)

PromptTemplate input variables: ['context', 'question']


## Part 4. Assemble the Baseline RAG Chain

Now you will assemble the complete RAG chain that you built in the Build Your First RAG System activity. The components are the same; only the corpus, the prompt, and the queries are different.

First, initialize the LLM.

In [10]:
# Do not remove or edit this cell

llm = ChatOpenAI(model="gpt-4o")

**Task**: Write a function called `format_docs` that takes a list of `Document` objects and returns a single string containing their `page_content`, separated by two newlines (`\n\n`).

In [11]:
# YOUR CODE HERE
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
# END OF YOUR CODE

**Task**: Assemble the baseline RAG chain. Save it to a variable called `rag_chain`. The chain should use:
- A dictionary mapping where `"context"` runs the retriever and pipes its output through `format_docs`, and `"question"` uses `RunnablePassthrough()`
- The `prompt` object created in Part 3
- The `llm` object initialized above
- A `StrOutputParser()` at the end

Use the pipe operator (`|`) to connect the components.

In [12]:
# YOUR CODE HERE
rag_chain = ({
    "context": retriever | format_docs, "question": RunnablePassthrough()
    } | prompt | llm | StrOutputParser()
)
# END OF YOUR CODE

Now let's test the chain on three customer queries. These are written to feel like what real shoppers actually type into a chat box. Pay attention to how the chain handles each one. They are deliberately different from each other:

- **Query 1** is short and underspecified, the kind of thing someone types when they have a quick question in their head.
- **Query 2** is a specific, scenario-based question about a real Marlowe & Finch policy.
- **Query 3** is a specific question that tests a particular edge case in Marlowe & Finch's policies.

In [13]:
# Do not remove or edit this cell

baseline_queries = [
    "is the tent waterproof",
    "can i return a tent i used on a weekend trip",
    "do you ship to australia"
]

baseline_answers = {}

for q in baseline_queries:
    answer = rag_chain.invoke(q)
    baseline_answers[q] = answer
    print(f"Q: {q}")
    print(f"A: {answer}\n")
    print("-" * 60)

Q: is the tent waterproof
A: The Trailhead 2 tent has a 2,000 mm hydrostatic head rating on both the fly and the floor, with fully taped seams, which means it performs well in steady rain and moderate wind. However, it is not rated for sustained heavy rain, alpine storms, or snow loading. For more extreme conditions, consider the Summit 4 four-season tent.

------------------------------------------------------------
Q: can i return a tent i used on a weekend trip
A: I'm sorry, but items that have been used outdoors are not eligible for return, even if they appear clean. If your tent has a manufacturing defect, it may be covered under our Trail-Tested Warranty. For further assistance, please contact our support team at support@marloweandfinch.com.

------------------------------------------------------------
Q: do you ship to australia
A: I'm sorry, but I don't have that information available. Please contact our support team at support@marloweandfinch.com for further assistance.

-----

**Task**: Review the three answers above. In the markdown cell below, answer the following:

1. How did the chain handle each of the three queries? Be specific: which ones produced clean answers, and which ones had problems? Were there any queries that the chain should have been able to answer but did not answer correctly? 
2. If a customer support manager saw these three answers, which one would they be most uncomfortable publishing on the website, and why?

1. **Performance Across Customer Queries:**
Query 1 ("is the tent waterproof") produced a clean, detailed answer by pulling specific product specs like hydrostatic head ratings and weather limitations for the Trailhead 2 tent. Query 2 ("can i return a tent i used on a weekend trip") generated an accurate response based on the return policy for outdoor-used items, offering warranty guidance alongside support contact info. Query 3 ("do you ship to australia") failed completely, returning a fallback response stating the information was unavailable. The chain should have answered Query 3 correctly because international shipping policies are standard knowledge base documentation.

2. **Customer Support Manager Concerns:**
A customer support manager would feel most uncomfortable publishing the answer to Query 3 ("do you ship to australia"). Presenting a total lack of basic shipping location data directly to a international customer would create friction, and forces unnecessary support ticket volume over a fundamental policy question.

## Part 5. Add Query Rewriting and Compare

Short, casual queries are a known weakness of vector retrieval. A query like "is the tent waterproof" is short and underspecified compared to the full-sentence documentation in the knowledge base.

In the query transformation activity, you saw that one way to address this is to rewrite the query into a longer, more complete version before sending it to the retriever. In this part, you will add a query rewriting step in front of the baseline chain and compare the results.

**Task**: Create a variable named `rewrite_prompt_template` that contains a prompt for rewriting a customer query. This prompt will be combined with the actual customer query using `.format()` in the next cell. Use the placeholder `{short_query}` where the customer's original query should appear.

Your prompt should:

1. Explain the task (rewrite a short customer query into a more complete version)
2. Make it clear that the rewritten query should preserve the customer's intent
3. Ask for exactly one rewritten query as output, not a list or numbered alternatives

*Note*: Because this template contains a curly-brace placeholder, define it as a regular string now and use `.format()` later, rather than as an f-string directly.

*Tip*: This is similar to the query rewriting work you did in the Improve RAG Retrieval Through Query Transformation activity, but there you asked for multiple alternatives. Here you want exactly one rewritten query so you can drop it straight into the existing retriever.

In [14]:
# YOUR CODE HERE
rewrite_prompt_template = """You are an AI assistant helping a customer support system for an outdoor gear company called Marlowe & Finch. 
Your task is to take a short or vague customer query and rewrite it into a complete, clear, and detailed question suitable for searching a technical knowledge base.

Guidelines:
- Preserve the original intent of the customer's question.
- Do not make assumptions or add unmentioned requirements.
- Output ONLY the single rewritten query string and nothing else (no introductory text or numbered options).

Customer Query: {short_query}"""

# END OF YOUR CODE

The cell below uses your `rewrite_prompt_template` to rewrite each of the three queries from Part 4. To keep things fast and consistent, we rewrite each query once and save the results in a dictionary called `rewritten_queries`. That way, when we compare baseline answers to rewritten-query answers, we are using the same rewrites throughout.

In [15]:
# Do not remove or edit this cell

rewritten_queries = {}

for q in baseline_queries:
    filled_prompt = rewrite_prompt_template.format(short_query=q)
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": filled_prompt}]
    )
    rewritten_queries[q] = response.choices[0].message.content.strip()
    print(f"Original:  {q}")
    print(f"Rewritten: {rewritten_queries[q]}\n")

Original:  is the tent waterproof
Rewritten: Is the Marlowe & Finch tent waterproof and what level of water resistance does it provide?

Original:  can i return a tent i used on a weekend trip
Rewritten: Can I return a tent that I purchased from Marlowe & Finch after using it on a weekend trip?

Original:  do you ship to australia
Rewritten: Does Marlowe & Finch offer shipping services to Australia for outdoor gear products?



Now let's run the full pipeline (the rewritten query through the existing RAG chain) and compare the answers side by side with the baseline.

In [16]:
# Do not remove or edit this cell

for q in baseline_queries:
    rewritten = rewritten_queries[q]
    rewritten_answer = rag_chain.invoke(rewritten)

    print(f"Original query: {q}")
    print(f"Rewritten query: {rewritten}\n")
    display(Markdown(f"**Baseline answer:** {baseline_answers[q]}"))
    display(Markdown(f"**Rewritten-query answer:** {rewritten_answer}"))
    print("=" * 60)

Original query: is the tent waterproof
Rewritten query: Is the Marlowe & Finch tent waterproof and what level of water resistance does it provide?



**Baseline answer:** The Trailhead 2 tent has a 2,000 mm hydrostatic head rating on both the fly and the floor, with fully taped seams, which means it performs well in steady rain and moderate wind. However, it is not rated for sustained heavy rain, alpine storms, or snow loading. For more extreme conditions, consider the Summit 4 four-season tent.

**Rewritten-query answer:** Our Trailhead 2 tent has a 2,000 mm hydrostatic head rating on both the fly and the floor, with fully taped seams, making it suitable for steady rain and moderate wind. However, it is not designed for sustained heavy rain, alpine storms, or snow loading. For more challenging conditions, we recommend considering the Summit 4 four-season tent. If you have any more questions, feel free to reach out.

Original query: can i return a tent i used on a weekend trip
Rewritten query: Can I return a tent that I purchased from Marlowe & Finch after using it on a weekend trip?



**Baseline answer:** I'm sorry, but items that have been used outdoors are not eligible for return, even if they appear clean. If your tent has a manufacturing defect, it may be covered under our Trail-Tested Warranty. For further assistance, please contact our support team at support@marloweandfinch.com.

**Rewritten-query answer:** I'm sorry, but items that have been used outdoors are not eligible for return, even if they appear clean. If the tent you purchased has a manufacturing defect, it might be covered under our Trail-Tested Warranty. You can start a warranty claim by emailing warranty@marloweandfinch.com with your order number, photos of the issue, and a brief description of how the issue occurred. For further assistance, please contact our support team at support@marloweandfinch.com.

Original query: do you ship to australia
Rewritten query: Does Marlowe & Finch offer shipping services to Australia for outdoor gear products?



**Baseline answer:** I'm sorry, but I don't have that information available. Please contact our support team at support@marloweandfinch.com for further assistance.

**Rewritten-query answer:** I'm sorry, but I don't have that information available. Please contact our support team at support@marloweandfinch.com for further assistance.

**Task**: Review the side-by-side comparison above. In the markdown cell below, answer the following:

1. On which of the three queries did query rewriting visibly change the answer? Did it improve the answer, make it worse, or leave it about the same?
2. Look at the rewritten queries themselves. Are they faithful to the original intent, or did the rewriting step change what the customer was asking about?
3. Query rewriting adds an extra LLM call before every retrieval, which costs both money and a bit of latency. Based on what you saw, is the improvement worth the added cost for this use case? When would you turn it on, and when would you leave it off?

1. **Impact on Answers:**
Query rewriting visibly altered the answers for the first two queries while leaving the third unchanged.
* Query 1 ("is the tent waterproof"): The rewritten answer shifted tone slightly to a first-person support persona ("Our Trailhead 2 tent...", "feel free to reach out") while keeping core technical specs, leaving overall quality about the same.
* Query 2 ("can i return a tent i used on a weekend trip"): The rewritten query produced a noticeably better answer by expanding warranty instructions to include `warranty@marloweandfinch.com`, required photos, and order submission steps.
* Query 3 ("do you ship to australia"): Both baseline and rewritten attempts produced identical fallback failure messages, leaving answer quality unchanged.

2. **Faithfulness to User Intent:**
The rewritten queries strictly preserved the customer's core intent. Each rewrite simply added explicit brand context ("Marlowe & Finch") and expanded vague phrasing into complete product questions without introducing false assumptions or altering the user's goal.

3. **Cost-Benefit and Production Deployment:**
The minor answer improvement does not justify the added latency and financial cost of running a secondary LLM call for every simple question. 

Turn query rewriting off for direct, standard customer questions or well-indexed document setups. Turn query rewriting on when handling conversational chat histories where follow-up questions rely heavily on prior context, or when user inputs are fragmented, ambiguous, and missing essential search keywords.

## Part 6. Analysis

You have now built a complete RAG assistant for Marlowe & Finch and tested a query rewriting improvement on top of it. In this section, reflect on the system as a whole.

Answer the following questions in the markdown cell below:

1. **Explaining the system to the team**: Marlowe & Finch's customer support manager is not technical. They want to know, in plain language, why the assistant sometimes gives a great answer and sometimes gives a vague one. Using what you observed in this lab, write a short explanation (3-5 sentences) you could give the manager. Avoid complicated technical jargon. Do not assume they know what an embedding is.

2. **Biggest deployment risk**: Marlowe & Finch is considering putting this assistant live on the website as a first-line responder to customer questions. Based on what you observed in this lab, what is the single biggest risk of deploying this prototype as-is? Name one specific type of customer question that would likely cause the assistant to behave badly, and explain what you would recommend to the team before going live.

**Answers**

1. **Explaining the System to Management:**
The assistant works by searching our knowledge base documents to find text passages matching a customer's question. When a customer asks about explicit product specs like tent waterproofing, the search tool locates exact matching paragraphs and generates precise answers. Simple or unusual questions like shipping destinations often fail because our reference documents lack matching keywords, forcing the system to give a generic fallback message. Adding comprehensive support documentation directly improves the quality of these answers.

2. **Primary Deployment Risk:**
Deploying this assistant as-is risks giving customers confident misinformation when dealing with order-specific actions or account changes. A question like "can you cancel order #48201 for my trip tomorrow" causes failure because the chatbot only reads standard reference text instead of connecting to store management software. I recommend adding a human handoff workflow alongside explicit policy boundaries before publishing the system online.

## Part 7. Reflection: AI Usage

1. Did you use AI tools for this lab? If yes, which ones and at what points in your work? If no, briefly explain your reasoning.
2. If you used AI, describe one specific prompt that was useful and explain why it worked. If you did not use AI, walk through one part of the lab where you had to figure something out on your own and explain how you got there.
3. How did you verify that your work was correct? What would you look for to catch a mistake, whether it came from AI or from your own reasoning?
4. What is one thing you would do differently next time, either in how you approached the lab or in how you used (or did not use) AI?


Record your findings in the cell below.

### AI Tool Usage and Reflection

I used Claude during Part 5 to generate the query rewriting prompt template, and I relied on Claude throughout the lab to help reframe and articulate my analytical findings clearly.

Prompt used: "Write a system prompt template that takes a short user query and transforms it into a detailed question for vector search. It must output exactly one string, preserve user intent, and use the placeholder {short_query}."

This prompt worked effectively because setting rigid constraints forced the model to return a single clean string without conversational intro text or numbered options, which allowed seamless string formatting via `.format()` directly inside the loop.

I verified the retrieval pipeline by inspecting the `rewritten_queries` dictionary outputs against the original input strings. I checked that no extra markdown code blocks or quotes leaked into the dictionary values. I confirmed that brand context ("Marlowe & Finch") was added without altering the customer's core intent.

Next time, I will test various temperature settings on the query rewriter to observe how varying creativity levels impact retrieval accuracy in vector databases.